# Regression Adjustment: Channel Effect

In [59]:
from pathlib import Path
import pandas as pd

Export_DIR = Path('../data/powerbi_exports')
Export_DIR.mkdir(parents = True, exist_ok=True)

def export_csv(df: pd.DataFrame, filename: str):
    out = Export_DIR / filename
    df.to_csv(out, index = False, encoding='utf-8')
    print(f'[Exported] {out.resolve()}')

## 1. Objective and Analytical Motivation

The unadjusted two-group comparison (notebook 02) indicated a statistically significant difference in conversion rates between cellular and telephone contact methods. However, the contact method was recorded rather than randomly assigned, raising concerns about selection bias and confounding.

Therefore, this stage estimates the adjusted channel association by controlling for observable customer characteristics using logistic regression.

In [60]:
# Import Libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf


# Load data
df = pd.read_csv('../data/preprocessed-bank-data.csv')

df_reg = df[df['contact'] .isin(['cellular', 'telephone'])].copy() 

## 2. Methodology: Logistic Regression for Confounder Adjustment

Considering that the target variable (`y`) is binary outcome, apply logistic regression as confounder adjustment methodology. This model controls for observable confounders such as age, job, marital status, loan status, prior campaign interactions, and macroeconomic variables to isolate the conditional effect of the contact method.

### 2.1 Model Specification

conversion ~ contact + age + job + marital + education + loan + housing + previous + month + macro_vars

- contact: main variable of interest
- other variables: control covariates (observable confounders)

*`macro_vars` refer to external macroeconomic indicators that may influence overall conversion likelihood independently of customer characteristics.*

macro_vars

- emp.var.rate: employment variation rate (economic labor market condition indicator)
- cons.price.idx: consumer price index
- cons.conf.idx: consumer confidence index
- euribor3m: 3-month Euribor rate
- nr.employed: total employment level in the economy (macro-level indicator)

In [61]:
df_reg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  campaign        41188 non-null  int64  
 11  pdays           41188 non-null  int64  
 12  previous        41188 non-null  int64  
 13  poutcome        41188 non-null  object 
 14  emp.var.rate    41188 non-null  float64
 15  cons.price.idx  41188 non-null  float64
 16  cons.conf.idx   41188 non-null  float64
 17  euribor3m       41188 non-null 

In [62]:
print(df_reg['housing'].unique())
print(df_reg['loan'].unique())

['no' 'yes' 'unknown']
['no' 'yes' 'unknown']


Although `housing` and `loan` were documented as 'binary', actual data contains 'unknown' and treat as categorical variables. Thus, categorical variables are `job`, `marital`, `education`, `month`, `housing`, `loan`.

In [63]:
# Define Target Variable
df_reg.rename(columns={'y':'conversion'}, inplace = True)

df_reg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  campaign        41188 non-null  int64  
 11  pdays           41188 non-null  int64  
 12  previous        41188 non-null  int64  
 13  poutcome        41188 non-null  object 
 14  emp.var.rate    41188 non-null  float64
 15  cons.price.idx  41188 non-null  float64
 16  cons.conf.idx   41188 non-null  float64
 17  euribor3m       41188 non-null 

`y` represents the result whether client subscribed or not, so rename it as `conversion`.


## 3. Understanding Confounding

A confounder is a variable that influences both:

1) Which channel was used to contact the customer (contact method), and
2) The outcome (conversion).

Because the contact channel was not randomly assigned, certain customer characteristics may have influenced which channel was used.

For example:
- Younger or digitally active customers may be more likely to be contacted via cellular.
- Customers with prior campaign engagement may have higher inherent conversion probability.

If these variables are not controlled for, the estimated channel association captures both:
- The channel effect itself
- AND systematic customer differences

This leads to biased estimation (confounding bias).

Therefore, a baseline confounder adjustment model is estimated using logistic regression to isolate the conditional association of contact while holding observable covariates constant.


### 3.1 Baseline Confounder Adjustment Model

To isolate the conditional association of the `contact` method, a logistic regression model is fitted while controlling for observable confounders.

The model specification is:
$$\text{conversion(y)} \sim \text{contact} + X$$

Where:

- `contact` is the exposure of interest (cellular vs telephone),
- `X` includes customer characteristics and macroeconomic indicators.

By including these covariates, the model compares customers with similar observable profiles across contact channels.

Thus, the estimated coefficient for `contact` reflects the adjusted channel association rather than raw group differences.

In [64]:
#   0/1 지시변수는 passthrough로 두어 exp(beta)가 telephone→cellular 오즈비가 되게 함.
#   다른 공변량 계수는 변하지 않음(최대 변화 5e-15). AUC 0.792→0.791, Accuracy 0.892→0.893.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Define Target
y = df_reg['conversion']

# Channel indicator (0/1). Kept unscaled so exp(beta) is the telephone -> cellular odds ratio
df_reg['contact_binary'] = (df_reg['contact'] == 'cellular').astype(int)

# Feature selection
feature_cols = ['contact_binary', 'age', 'previous',
                'job','marital','education', 'housing', 'loan', 'month',
                'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

X = df_reg[feature_cols].copy()

categorical_cols = ['job', 'marital','education','housing','loan', 'month']
numeric_cols = ['age','previous',
                'emp.var.rate', 'cons.price.idx', 'cons.conf.idx','euribor3m', 'nr.employed']

# Preprocessing
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocess = ColumnTransformer([
    ('bin', 'passthrough', ['contact_binary']),
    ('num', numeric_pipe, numeric_cols),
    ('cat', categorical_pipe, categorical_cols)
])

model = LogisticRegression(max_iter=2000)

clf = Pipeline([
    ('preprocess', preprocess),
    ('model', model)
])

# Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify = y)

clf.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('bin', 'passthrough',
                                                  ['contact_binary']),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'previous',
                                                   'emp.var.rate',
                                                   'cons.price.idx',
                                                   'cons.conf.idx', 'euribor3m',
                                                   'nr.employed']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['job', 'marital',
                                                   'education', 'housing',
                                                   'loan', 'month'])])),
                ('model', LogisticRegression(max_iter=2000))])

## 4. Model Estimation

After training the baseline confounder adjustment model, we:

1. Extract coefficient estimates
2. Convert coefficients to odds ratios 
3. Assess statistical relevance of the contact effect
4. Perform a basic predicitive sanity check

The primary objective is inference (adjusted effect estimation), not pure predictive optimisation.

### 4.1 Extract Coefficients & Odd Ratios

In [65]:
# Retrieve feature names
feature_names = clf.named_steps['preprocess'].get_feature_names_out()

coef = clf.named_steps['model'].coef_.ravel()

coef_table = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coef,
    'odds_ratio': np.exp(coef)
}).sort_values('odds_ratio', ascending=False)

coef_table[coef_table['feature'].str.contains('contact', case= False, na=False)]

,feature,coefficient,odds_ratio
0,bin__contact_binary,0.754349,2.126226



After fitting the baseline logistic regression model with confounder adjustment, we extract the estimated coefficients and convert them to odds ratios for interpretability.

For the contact variable (`bin__contact_binary`), the pipeline estimate is:

- Coefficient (β) = 0.7543
- Odds Ratio (exp(β)) ≈ 2.13

This suggests that, holding other variables constant, customers contacted via cellular have roughly **2.1 times the conversion odds** of customers contacted via telephone.

Note: the pipeline's `LogisticRegression` applies mild L2 regularisation by default. The unregularised GLM in Section 4.2, used for formal inference, gives β = 0.7685 and OR = 2.16. The two agree closely.

### 4.2 Assess statistical significance (p-value / CI)

In [66]:
# Preprocess Result
X_train_processed = clf.named_steps['preprocess'].transform(X_train)

# Dense + Float covert to prevent ValueError
if hasattr(X_train_processed, 'toarray'):
    X_dense = X_train_processed.toarray()
else:
    X_dense = np.asarray(X_train_processed)

X_dense = X_dense.astype(np.float64)

# Get feature names
feature_names = clf.named_steps['preprocess'].get_feature_names_out()
X_df = pd.DataFrame(X_dense, columns=feature_names, index=y_train.index)

# Add a constant
X_df = sm.add_constant(X_df, has_constant='add')

# GLM (Binomial)
glm = sm.GLM(y_train, X_df, family=sm.families.Binomial())
result = glm.fit()

print(result.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             conversion   No. Observations:                32950
Model:                            GLM   Df Residuals:                    32908
Model Family:                Binomial   Df Model:                           41
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -9379.4
Date:                Mon, 17 Aug 2026   Deviance:                       18759.
Time:                        14:55:04   Pearson chi2:                 3.30e+04
No. Iterations:                     6   Pseudo R-squ. (CS):             0.1261
Covariance Type:            nonrobust                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
cons

In [67]:

contact_term = [t for t in result.params.index if 'contact_binary' in t][0]
ci_coef = result.conf_int().loc[contact_term]

print(f"Adjusted OR: {np.exp(result.params[contact_term]):.2f}")
print(f"Odds Ratio 95% CI: [{np.exp(ci_coef[0]):.2f}, {np.exp(ci_coef[1]):.2f}]")
print(f"p-value: {result.pvalues[contact_term]:.2e}")

# Crude (unadjusted) odds ratio, for the Section 5 comparison
ct = pd.crosstab(df_reg['contact'], df_reg['conversion'])
crude_or = (ct.loc['cellular', 1] * ct.loc['telephone', 0]) / (ct.loc['cellular', 0] * ct.loc['telephone', 1])
print(f"Crude OR (no adjustment): {crude_or:.2f}")

Adjusted OR: 2.16
Odds Ratio 95% CI: [1.87, 2.49]
p-value: 6.69e-26
Crude OR (no adjustment): 3.13


To evaluate whether the observed contact effect is statistically reliable, we examine the p-value and confidence interval from the GLM output.

For `bin__contact_binary`:

- p-value < 0.001
- 95% CI (coefficient): [0.625, 0.912]
- 95% CI (odds ratio): approximately [1.87, 2.49]

The extremely small p-value indicates that the likelihood of observing this association by random chance is negligible. From a business perspective, even the lower bound of the confidence interval implies an **87% increase in conversion odds**, so the conclusion does not depend on reading the estimate optimistically.

### 4.3 Perform a Basic Predictive Sanity Check

Although the primary objective of this model is inference rather than prediction, it is important to confirm that the model demonstrates reasonable predictive performance. We evaluate out-of-sample performance using AUC and accuracy metrics.

In [68]:
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

# Predict probabilities
y_pred_prob = clf.predict_proba(X_test)[:, 1]

# Predict class
y_pred = clf.predict(X_test)

# Metrics
auc = roc_auc_score(y_test, y_pred_prob)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("AUC:", round(auc, 3))
print("Accuracy:", round(acc, 3))
print("Confusion Matrix:\n", cm)

AUC: 0.791
Accuracy: 0.893
Confusion Matrix:
 [[7212   98]
 [ 787  141]]


The model achieves an AUC of 0.79, indicating reasonable discrimination ability between converters and non-converters. While the predictive performance is not the primary objective, the model demonstrates sufficient stability to support inference on coefficient estimates.



## 5. Unadjusted vs Adjusted Comparison

| Method | Estimate | Interpretation |
|--------|----------|----------------|
| Unadjusted two-group comparison (notebook 02) | +9.5%p (crude OR 3.13) | Raw difference |
| Logistic Regression (Adjusted) | OR = 2.16 (95% CI: 1.87 - 2.49) | Adjusted association |

The unadjusted comparison shows a raw conversion lift of approximately 9.5 percentage points between cellular and telephone, which corresponds to a crude odds ratio of 3.13. This estimate does not account for differences in customer composition or campaign timing.

By contrast, logistic regression isolates the conditional (partial) association between contact type and conversion while controlling for the other covariates in the model. The adjusted odds ratio of 2.16 means that, holding other variables constant, customers contacted via cellular have roughly 2.2 times the conversion odds of customers contacted via telephone.

Adjustment therefore absorbs about a third of the raw gap on the log-odds scale (3.13 → 2.16). A clear positive association survives, but part of the raw difference reflected who was contacted on each channel rather than the channel itself, which is exactly why the unadjusted gap should not be read as a channel effect.

## 6. Robustness and Sensitivity

To assess the stability of the estimated contact effect, an interaction specification (contact x age) is fitted and the contact coefficient is compared across the two models.

The direction and magnitude of the contact effect remain consistent across the two specifications. A further check with a different control set that excludes macroeconomic variables is carried out in notebook 04, which reaches a similar adjusted estimate (OR 2.33 there vs 2.16 here).

In [69]:
# Model 2: Add interaction (contact x age)
X_inter = X_df.copy()
X_inter['contact_age_interaction'] = (
    X_inter['bin__contact_binary'] * X_inter['num__age']
)

glm_inter = sm.GLM(y_train, X_inter, family=sm.families.Binomial())

result_inter = glm_inter.fit()

print(result_inter.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             conversion   No. Observations:                32950
Model:                            GLM   Df Residuals:                    32907
Model Family:                Binomial   Df Model:                           42
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -9379.3
Date:                Mon, 17 Aug 2026   Deviance:                       18759.
Time:                        14:55:05   Pearson chi2:                 3.30e+04
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1261
Covariance Type:            nonrobust                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
cons

To assess the stability of the contact effect, an interaction term between contact and age was introduced.

The interaction term was not statistically significant (p = 0.612), and the magnitude of the contact coefficient remained virtually unchanged.

This suggests that the estimated contact effect is not meaningfully moderated by age and remains stable across alternative specifications.


## 7. Interpretation and Business Implications

The contact channel remains statistically significant after adjustment. An adjusted odds ratio of 2.16 means cellular contact is associated with roughly twice the conversion odds of telephone contact. From a strategic perspective, prioritizing cellular contact may enhance conversion efficiency, provided operational constraints and channel costs are carefully evaluated.

These findings support further validation through structured experimentation. A randomised holdout would identify the causal effect that this observational estimate cannot.

## 8. Limitations

- The analysis is based on observational data.
- Potential unobserved confounding cannot be fully ruled out.
- The results indicate strong conditional association but do not establish causality.
- Experimental or quasi-experimental validation is recommended.

## 9. Executive Summary

Cellular contact is associated with roughly 2.2 times the conversion odds of telephone (adjusted OR 2.16, 95% CI 1.87-2.49) after adjusting for customer and macroeconomic factors. The unadjusted odds ratio is 3.13, so adjustment absorbs about a third of the raw gap while a clear positive association survives.

The analysis does not provide causal evidence: contact method was recorded, not assigned. The size and stability of the association justify strategic consideration and further experimental validation.

### Power BI Result Table Export

In [70]:
from pathlib import Path
import pandas as pd

Export_DIR = Path('../data/powerbi_exports')
Export_DIR.mkdir(parents = True, exist_ok=True)

def export_csv(df: pd.DataFrame, filename: str):
    out = Export_DIR / filename
    df.to_csv(out, index = False, encoding='utf-8')
    print(f'[Exported] {out.resolve()}')

In [71]:
def glm_result_to_table(res, model_name: str):
    params = res.params
    bse = res.bse
    pvals = res.pvalues
    conf = res.conf_int() # [low, high] on log-odds scale

    out = pd.DataFrame({
        'model': model_name,
        'term': params.index.astype(str),
        'coefficient': params.values.astype(float),
        'standard_error': bse.values.astype(float),
        'p_value': pvals.values.astype(float),
        'ci_low_coef' : conf[0].values.astype(float),
        'ci_high_coef' : conf[1].values.astype(float)
    })

    # odds ratio + CI on OR scale
    out['odds_ratio'] = np.exp(out['coefficient'])
    out['ci_low_or'] = np.exp(out['ci_low_coef'])
    out['ci_high_or'] = np.exp(out['ci_high_coef'])

    # Core Terms
    out['is_contact_term'] = out['term'].str.contains('contact', case=False, na=False)

    return out

# Power BI Export: GLM coefficient tables
tbl_main = glm_result_to_table(result, 'glm_main')
export_csv(tbl_main, 'glm_main_coefficients.csv')

# If interaction model exists, export as well
if 'result_inter' in globals():
    tbl_inter = glm_result_to_table(result_inter, 'glm_contact_x_age')
    export_csv(tbl_inter, 'glm_interaction_coefficients.csv')


[Exported] /Users/gayoungdan/Documents/DS MS/Job Hunting 💼/marketing-campaign-casual-evaluation/data/powerbi_exports/glm_main_coefficients.csv
[Exported] /Users/gayoungdan/Documents/DS MS/Job Hunting 💼/marketing-campaign-casual-evaluation/data/powerbi_exports/glm_interaction_coefficients.csv
